In [ ]:
# %pip uninstall -y numpy pandas scikit-learn tqdm 

In [ ]:
# %pip install -q numpy==1.26.3 pandas==1.5.3 scikit-learn==1.5.2 tqdm dice_ml openml lime shap gower deap --force-reinstall

In [1]:
from experiments.core.data_fetcher import OpenMLFetcher
from experiments.core.path_utils import path_generator
from experiments.core.data_loader import TabularDataLoader
from experiments.core.data_preprocessing import PreprocessorBuilder
from experiments.models.classification_trainer import SklearnTabularModeler, dynamic_MLP_layers
from experiments.models.autoencoder_trainer import AutoencoderTrainer
from experiments.counterfactuals.cf_generator import CounterfactualGenerator
from experiments.analysis.result_reader import ResultReader

from joblib import Parallel, delayed
import multiprocessing
import pickle
from tqdm import tqdm

n_jobs = multiprocessing.cpu_count() - 2
print(f"Using {n_jobs} cores")

Using 10 cores


In [2]:
data_openml = {
    'adult': 45068, #48k, 15 features, >50k thu nhập
    'GiveMeSomeCredit': 46929, #150k, 11 features, vỡ nợ
    # 'Bank_Customer_Churn': 46911, # 10k, 11features, khách hàng rời bỏ
    # 'compas-two-years': 42192, # 5k, tái phạm tội
    # 'pima_diabetes': 37, # 9 feats, tiểu đường
    # 'Heart-Disease-Dataset-(Comprehensive)': 43672, #1k1, 12feats, bệnh tim
    'mushroom': 24, #23 cat feat, poision or edible, nấm độc
    'phoneme': 1489, # 5 numerical feat, phân loại âm vị
    # 'tokyo1': 40705, #45 numerical feat
}


PARAMETERS_GRID ={
    'RF':{'n_estimators': [50,100, 250, 500, 1000],
          'max_depth': [1,2,5,10, 25, None]
          },
    'ANN':{'hidden_layer_sizes':[]}
}

AE_GRID = {'learning_rate_init': [0.05,0.01,0.005,0.001,0.0005,0.0001,0.00005,0.00001]}

CF_WRAPPERS = [
            'monicemultiobjectivegower',
            'monicemultiobjectiveheom',
            'monicesparsprox',
            'moniceproxplaus',
            'monicesparsplaus',
            
            # 'nicenone',
            # 'nicespars',
            # 'niceprox',
            # 'niceplaus',
            
            # 'dicerandom',
            # 'moc',
            # 'geco',
            # 'lime',
            ]

In [3]:
def fetch_datasets(dataset_name, dataset_id):
    fetcher = OpenMLFetcher(dataset_name, dataset_id)
    fetcher.save()

In [4]:
def run_preprocessing(dataset_name):
    preprocessor = PreprocessorBuilder(dataset_name)
    preprocessor.build()

In [5]:
def run_modeling(dataset_name, model, grid):
    modeler= SklearnTabularModeler(dataset_name, model)
    if model == 'ANN':
        grid['hidden_layer_sizes']= dynamic_MLP_layers(dataset_name,10,1.5)
    modeler.grid_search(grid)
    modeler.save()
    modeler.save_stats()

In [6]:
def run_ae_modeling(dataset):
    ae_modeler = AutoencoderTrainer(dataset)
    ae_modeler.fit(AE_GRID)

In [7]:
def run_cf_experiment(dataset_name, model, cf_name):
    experiment = CounterfactualGenerator(dataset_name, model, cf_name)
    return experiment.run_experiment()

In [8]:
# run_cf_experiment('adult', 'ANN', 'monicemultiobjectivegower')

In [ ]:
# 1. Fetching, Preprocessing và AE Modeling
# Parallel(n_jobs=n_jobs)(
#     delayed(lambda dataset: (
#         print(f"========== {dataset} =========="),
#         fetch_datasets(dataset, data_openml[dataset]),
#         run_preprocessing(dataset),
#         run_ae_modeling(dataset)
#     ))(dataset)
#     for dataset in data_openml
# )

In [ ]:
# 2. Model Training
# for dataset in data_openml:
#     print(f"========== {dataset} ==========")
#     Parallel(n_jobs=n_jobs)(
#         delayed(lambda model: (
#             print(f"========== {model} =========="),
#             run_modeling(dataset, model, PARAMETERS_GRID[model]),
#             SklearnTabularModeler(dataset, model).display_stats()
#         ))(model)
#         for model in ['RF', 'ANN']
#     )

In [ ]:
# 3. Counterfactual Experiments
tasks = [
    (dataset, model, cf_name)
    for cf_name in CF_WRAPPERS
    for dataset in data_openml
    for model in ['RF', 'ANN']
]

Parallel(n_jobs=n_jobs)(
    delayed(run_cf_experiment)(dataset, model, cf_name)
    for dataset, model, cf_name in tqdm(tasks, desc="Processing CF_WRAPPERS")
)

Processing CF_WRAPPERS: 100%|██████████| 40/40 [51:20<00:00, 77.02s/it] 


In [ ]:
reader = ResultReader(
    dataset_names=data_openml,
    models=['ANN', 'RF'],
    cf_names=CF_WRAPPERS
)

Loaded and preprocessed 200 results from ./data/adult/results/ANN_monicemultiobjectivegower.pkl
Loaded and preprocessed 200 results from ./data/adult/results/ANN_nicenone.pkl
Loaded and preprocessed 200 results from ./data/adult/results/ANN_dicerandom.pkl
Loaded and preprocessed 200 results from ./data/adult/results/ANN_moc.pkl
Loaded and preprocessed 200 results from ./data/compas-two-years/results/ANN_monicemultiobjectivegower.pkl
Loaded and preprocessed 200 results from ./data/compas-two-years/results/ANN_nicenone.pkl
Loaded and preprocessed 200 results from ./data/compas-two-years/results/ANN_dicerandom.pkl
Loaded and preprocessed 200 results from ./data/compas-two-years/results/ANN_moc.pkl


In [ ]:
reader.analyze_results()

Generating analysis tables...
Processing ANN - coverage
Saved ./tables/ANN_coverage.csv
Processing ANN - validity
Saved ./tables/ANN_validity.csv
Processing ANN - time
Saved ./tables/ANN_time.csv
Processing ANN - sparsity
Saved ./tables/ANN_sparsity.csv
Processing ANN - confidence
Saved ./tables/ANN_confidence.csv
Processing ANN - entropy
Saved ./tables/ANN_entropy.csv
Processing ANN - gower_proximity
Saved ./tables/ANN_gower_proximity.csv
Processing ANN - heom_proximity
Saved ./tables/ANN_heom_proximity.csv
Processing ANN - ae_loss
Saved ./tables/ANN_ae_loss.csv
Processing ANN - diversity
Saved ./tables/ANN_diversity.csv
Processing ANN - 5NN_distance
Saved ./tables/ANN_5NN_distance.csv
Processing ANN - IM1
Saved ./tables/ANN_IM1.csv
Processing ANN - IM2
Saved ./tables/ANN_IM2.csv


{'ANN': {'coverage':                  monicemultiobjectivegower nicenone dicerandom    moc
  adult                                100.0    100.0       99.5  100.0
  compas-two-years                     100.0    100.0      100.0  100.0,
  'validity':                  monicemultiobjectivegower nicenone dicerandom        moc
  adult                                100.0    100.0  91.916667  99.833333
  compas-two-years                     100.0    100.0  80.166667      100.0,
  'time':                  monicemultiobjectivegower  nicenone dicerandom       moc
  adult                             1.054561   0.01099   0.237307  3.052571
  compas-two-years                  0.125432  0.002224   0.144268  1.861303,
  'sparsity':                  monicemultiobjectivegower nicenone dicerandom       moc
  adult                             3.774167    4.445   1.605621  3.204508
  compas-two-years                  2.098333     1.89   1.589397     3.255,
  'confidence':                  monicemultiobje

In [ ]:
reader.statistical_analysis()

Starting feature change analysis...
Analyzing adult - ANN...
Generated: ./stats/ANN-adult-0-1.md
Generated: ./stats/ANN-adult-1-0.md
Analyzing compas-two-years - ANN...
Generated: ./stats/ANN-compas-two-years-1-0.md
Generated: ./stats/ANN-compas-two-years-0-1.md
Feature change analysis completed!


In [ ]:
def load_results(dataset_name, model, cf_name):
    paths = path_generator(dataset_name, model, cf_name)
    with open(paths['results'], 'rb') as f:
        results = pickle.load(f)
    
    return results

In [ ]:
ret = load_results('adult', 'ANN', 'monicemultiobjectiveheom')
len(ret)

In [ ]:
ret